In [ ]:
#  !pip install -U langchain langchain-openai openai python-dotenv requests


In [4]:
# Load API keys
with open(r"C:\mindful-ai\sapient-ds\2024\presentation\week-06\openai-api-key-purushotham.txt") as f:
    openai_api_key = f.read().strip()
openweathermap_api_key = "94d2513070b90c306f61b4a02b3cad38"  

In [3]:
!curl -k "https://api.openweathermap.org/data/2.5/weather?q=London&appid=94d2513070b90c306f61b4a02b3cad38"

{"coord":{"lon":-0.1257,"lat":51.5085},"weather":[{"id":804,"main":"Clouds","description":"overcast clouds","icon":"04d"}],"base":"stations","main":{"temp":291.25,"feels_like":291.31,"temp_min":290.64,"temp_max":292.11,"pressure":1016,"humidity":84,"sea_level":1016,"grnd_level":1012},"visibility":10000,"wind":{"speed":6.69,"deg":230,"gust":11.83},"clouds":{"all":100},"dt":1758114266,"sys":{"type":2,"id":2075535,"country":"GB","sunrise":1758087523,"sunset":1758132687},"timezone":3600,"id":2643743,"name":"London","cod":200}


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100   527  100   527    0     0   1846      0 --:--:-- --:--:-- --:--:--  1875


In [5]:
# ===== LangChain imports (modern) =====
# If your installation uses slightly different module paths, see notes below.
try:
    from langchain_core.tools import tool        # recommended decorator
    from langchain_openai import OpenAI          # LLM wrapper
    from langchain.agents.react.agent import create_react_agent
    from langchain.agents import AgentExecutor
    from langchain_core.prompts import PromptTemplate
except Exception:
    # Fallback imports if your version differs:
    from langchain.tools import tool
    from langchain.llms import OpenAI
    from langchain.agents.react.agent import create_react_agent
    from langchain.agents import AgentExecutor
    from langchain.prompts import PromptTemplate

In [23]:
import requests
# ===== 1) Tool: fetch weather =====
@tool("get_current_weather", return_direct=True)
def get_current_weather(city: str) -> str:
    """Return current weather summary for a city. Input: city name (e.g. 'London')."""
    print("CITY -> ", city)
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {"q": city, "appid": openweathermap_api_key}
    resp = requests.get(url, params=params, timeout=10)
    if resp.status_code != 200:
        return f"ERROR: {resp.status_code} - {resp.text}"
    data = resp.json()
    desc = data["weather"][0]["description"]
    temp = data["main"]["temp"]
    feels = data["main"].get("feels_like", temp)
    wind = data.get("wind", {}).get("speed", 0)
    return (f"{city}: {desc}. Temperature {temp:.1f}°C (feels like {feels:.1f}°C). "
            f"Wind {wind} m/s.")

get_current_weather('London')

CITY ->  London


'London: overcast clouds. Temperature 291.4°C (feels like 291.5°C). Wind 6.69 m/s.'

In [24]:
# ===== 2) LLM (LangChain wrapper) =====
llm = OpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=openai_api_key)  # pick model you have access to

In [25]:
from langchain import hub
# ===== 3A) Standard ReAct prompt from LangChain hub =====
react_prompt_hub = hub.pull("hwchase17/react")

c:\Users\mindf\anaconda3\Lib\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [26]:
# ===== 3B) Custom ReAct prompt (explicitly including required vars) =====
react_prompt_custom = PromptTemplate(
    input_variables=["input", "tools", "tool_names", "agent_scratchpad"],
    template="""You are an AI assistant that can reason and act using the tools provided.

TOOLS:
{tools}

You can call any of these tools when necessary. 
When responding, use this format:

Question: the input question
Thought: your reasoning
Action: the tool to use, one of [{tool_names}]
Action Input: the input to the tool
Observation: the result of running the tool
... (this Thought/Action/Action Input/Observation can repeat)
Thought: I now know the final answer
Final Answer: the final answer to the original input

Begin!

Question: {input}
{agent_scratchpad}
"""
)



In [27]:
# ===== 4) Create agents =====
tools = [get_current_weather]

In [28]:
# Option A: Using LangChain hub prompt
agent_hub = create_react_agent(llm, tools, react_prompt_hub)
executor_hub = AgentExecutor(agent=agent_hub, tools=tools, verbose=True)

In [29]:
# Option B: Using custom prompt
agent_custom = create_react_agent(llm, tools, react_prompt_custom)
executor_custom = AgentExecutor(agent=agent_custom, tools=tools, verbose=True)

In [32]:
# ===== 5) Run agents =====
if __name__ == "__main__":
    print("=== Using Hub Prompt ===")
    response_hub = executor_hub.invoke({"input": "What is the weather like in London,GB?"})
    print("Agent Output:", response_hub["output"])

    print("\n=== Using Custom Prompt ===")
    response_custom = executor_custom.invoke({"input": "What is the weather like in Bangalore,IN?"})
    print("Agent Output:", response_custom["output"])

=== Using Hub Prompt ===


> Entering new AgentExecutor chain...
 I need to find out the current weather in London,GB.
Action: get_current_weather
Action Input: London,GBCITY ->  London,GB
London,GB: overcast clouds. Temperature 291.4°C (feels like 291.5°C). Wind 6.69 m/s.


> Finished chain.
Agent Output: London,GB: overcast clouds. Temperature 291.4°C (feels like 291.5°C). Wind 6.69 m/s.

=== Using Custom Prompt ===


> Entering new AgentExecutor chain...
Thought: I need to find out the current weather in Bangalore, India.
Action: get_current_weather
Action Input: Bangalore,INCITY ->  Bangalore,IN
Bangalore,IN: broken clouds. Temperature 298.3°C (feels like 298.9°C). Wind 0 m/s.


> Finished chain.
Agent Output: Bangalore,IN: broken clouds. Temperature 298.3°C (feels like 298.9°C). Wind 0 m/s.
